# Chương 7: Tối Ưu Hóa Liên Tục (Continuous Optimization)

Huấn luyện mô hình ML thực chất là **tìm tham số tốt nhất** cho hàm mục tiêu. Chương này trình bày các thuật toán tối ưu hóa số trị — công cụ cốt lõi để làm điều đó.

**Hai nhánh chính:**
- **Tối ưu không ràng buộc (Mục 7.1):** Gradient descent và các biến thể
- **Tối ưu có ràng buộc (Mục 7.2):** Lagrange multipliers
- **Tối ưu lồi (Mục 7.3):** Bảo đảm tìm được cực tiểu toàn cục

**Quy ước:** Hầu hết hàm mục tiêu trong ML được **tối thiểu hóa** (minimization). Gradient chỉ hướng dốc lên → đi ngược gradient để tìm thung lũng.

---

## Ví dụ mở đầu — Tại sao cần thuật toán tối ưu?

Cho hàm:
$$\ell(x) = x^4 + 7x^3 + 5x^2 - 17x + 3 \tag{7.1}$$

Tìm cực tiểu bằng cách đặt gradient = 0:
$$\frac{d\ell}{dx} = 4x^3 + 21x^2 + 10x - 17 = 0 \tag{7.2}$$

Phương trình bậc 3 → 3 nghiệm. Phân biệt cực tiểu/cực đại bằng đạo hàm bậc 2:

$$\frac{d^2\ell}{dx^2} = 12x^2 + 42x + 10 \tag{7.3}$$

- $\frac{d^2\ell}{dx^2} > 0$ → **cực tiểu**
- $\frac{d^2\ell}{dx^2} < 0$ → **cực đại**

Hàm có cực tiểu toàn cục tại $x \approx -4.5$ và cực tiểu địa phương tại $x \approx 0.7$. Với đa thức bậc cao hơn (≥5), không có nghiệm giải tích — **bắt buộc phải dùng thuật toán số trị**.

> **Vấn đề cốt lõi:** Nếu xuất phát từ $x_0 = 0$, gradient descent dẫn đến cực tiểu địa phương $x \approx 0.7$, bỏ qua cực tiểu toàn cục. Đây là lý do **điểm khởi đầu** và **hàm lồi** quan trọng.

Bạn gửi tiếp nội dung các mục 7.1, 7.2, 7.3 nhé — mình sẽ tổng hợp chi tiết từng phần.

# 7.1 Tối Ưu Hóa bằng Gradient Descent

---

## Ý Tưởng Cốt Lõi

Bài toán tối ưu không ràng buộc:

$$\min_\mathbf{x} f(\mathbf{x}) \tag{7.4}$$

với $f: \mathbb{R}^d \to \mathbb{R}$ khả vi, không có nghiệm giải tích dạng đóng.

**Gradient descent** là thuật toán tối ưu **bậc nhất** (first-order) — chỉ dùng thông tin gradient. Ý tưởng: gradient $\nabla f(\mathbf{x}_0)$ chỉ hướng dốc lên nhanh nhất → đi **ngược chiều gradient** để giảm hàm mục tiêu nhanh nhất.

**Trực quan hình học:** Gradient vuông góc với các đường đồng mức (contour lines) của $f$. Đi ngược gradient = đi vuông góc với đường đồng mức, xuống thung lũng theo đường dốc nhất.

**Công thức cập nhật:**

$$\mathbf{x}_{i+1} = \mathbf{x}_i - \gamma_i \left(\nabla f(\mathbf{x}_i)\right)^\top \tag{7.6}$$

với step-size $\gamma_i \geq 0$. Với step-size phù hợp: $f(\mathbf{x}_0) \geq f(\mathbf{x}_1) \geq \cdots$ hội tụ về cực tiểu địa phương.

> **Tại sao dùng chuyển vị?** Gradient được định nghĩa là vectơ hàng ($1 \times d$), cần chuyển vị thành vectơ cột ($d \times 1$) để cộng với $\mathbf{x}_i \in \mathbb{R}^d$.

---

### Ví dụ 7.1 — Gradient Descent trên Hàm Bậc Hai 2D

Cho hàm:

$$f\!\left(\begin{bmatrix}x_1\\x_2\end{bmatrix}\right) = \frac{1}{2}\begin{bmatrix}x_1\\x_2\end{bmatrix}^\top\begin{bmatrix}2&1\\1&20\end{bmatrix}\begin{bmatrix}x_1\\x_2\end{bmatrix} - \begin{bmatrix}5\\3\end{bmatrix}^\top\begin{bmatrix}x_1\\x_2\end{bmatrix} \tag{7.7}$$

Gradient:

$$\nabla f = \begin{bmatrix}x_1\\x_2\end{bmatrix}^\top\begin{bmatrix}2&1\\1&20\end{bmatrix} - \begin{bmatrix}5\\3\end{bmatrix}^\top \tag{7.8}$$

Xuất phát từ $\mathbf{x}_0 = [-3,-1]^\top$, với $\gamma = 0.085$:

- $\mathbf{x}_0 = [-3, -1]^\top$
- $\mathbf{x}_1 = [-1.98, 1.21]^\top$
- $\mathbf{x}_2 = [-1.32, -0.42]^\top$
- ... tiếp tục hội tụ về cực tiểu

> **Vấn đề điều kiện xấu (ill-conditioning):** Khi ma trận có eigenvalue rất lớn và rất nhỏ (condition number $\kappa = \sigma_{max}/\sigma_{min}$ lớn), bề mặt hàm giống thung lũng hẹp và dài. Gradient descent **zigzag** qua lại giữa hai thành thung lũng, hội tụ rất chậm.

---

## 7.1.1 Kích Thước Bước (Step-size / Learning Rate)

Chọn step-size $\gamma$ là bài toán cân bằng quan trọng:

| Step-size | Hậu quả |
|---|---|
| Quá nhỏ | Hội tụ chậm |
| Quá lớn | Vượt qua cực tiểu (overshoot), có thể phân kỳ |
| Phù hợp | Hội tụ đơn điệu về cực tiểu |

**Hai heuristic thích ứng đơn giản:**
- $f(\mathbf{x}_{i+1}) > f(\mathbf{x}_i)$: step-size quá lớn → **hoàn tác bước, giảm** $\gamma$
- $f(\mathbf{x}_{i+1}) < f(\mathbf{x}_i)$: step có thể lớn hơn → **tăng** $\gamma$

Mặc dù bước "hoàn tác" tốn tài nguyên, chiến lược này **đảm bảo hội tụ đơn điệu**.

> **Condition number và tốc độ hội tụ:** Khi giải $Ax = b$ bằng gradient descent, tốc độ phụ thuộc vào condition number $\kappa = \sigma_{max}(A)/\sigma_{min}(A)$. $\kappa$ lớn → hội tụ chậm. Kỹ thuật **preconditioning** (thay $Ax=b$ bằng $P^{-1}(Ax-b)=0$) cải thiện condition number để tăng tốc hội tụ.

---

## 7.1.2 Gradient Descent với Momentum

**Vấn đề:** Gradient descent thuần túy zigzag trong thung lũng hẹp — mỗi bước đi gần vuông góc với bước trước, lãng phí.

**Giải pháp:** Thêm **bộ nhớ** — nhớ hướng đi trước và duy trì đà (momentum):

$$\mathbf{x}_{i+1} = \mathbf{x}_i - \gamma_i\left(\nabla f(\mathbf{x}_i)\right)^\top + \alpha \Delta\mathbf{x}_i \tag{7.11}$$

$$\Delta\mathbf{x}_i = \mathbf{x}_i - \mathbf{x}_{i-1} = \alpha\Delta\mathbf{x}_{i-1} - \gamma_{i-1}\left(\nabla f(\mathbf{x}_{i-1})\right)^\top \tag{7.12}$$

với $\alpha \in [0,1]$ là hệ số momentum.

> **Trực quan:** Như quả bóng nặng lăn xuống dốc — không thay đổi hướng đột ngột, duy trì đà theo hướng cũ. Momentum **triệt tiêu dao động** và **tăng tốc** theo hướng hội tụ nhất quán.

**Bonus:** Khi gradient nhiễu (noisy), momentum hoạt động như **moving average** — làm mịn gradient, giảm ảnh hưởng của nhiễu ngẫu nhiên.

---

## 7.1.3 Stochastic Gradient Descent (SGD)

### Bài toán tổng quát trong ML

Với $N$ điểm dữ liệu, hàm mục tiêu thường có dạng:

$$L(\boldsymbol{\theta}) = \sum_{n=1}^N L_n(\boldsymbol{\theta}) \tag{7.13}$$

Ví dụ negative log-likelihood trong hồi quy:

$$L(\boldsymbol{\theta}) = -\sum_{n=1}^N \log p(y_n \mid \mathbf{x}_n, \boldsymbol{\theta}) \tag{7.14}$$

**Batch gradient descent** — dùng toàn bộ dữ liệu:

$$\boldsymbol{\theta}_{i+1} = \boldsymbol{\theta}_i - \gamma_i\sum_{n=1}^N \left(\nabla L_n(\boldsymbol{\theta}_i)\right)^\top \tag{7.15}$$

**Vấn đề:** Với $N$ triệu điểm dữ liệu, mỗi bước cập nhật tốn kém vô cùng.

### SGD — Dùng tập con ngẫu nhiên (mini-batch)

**Ý tưởng then chốt:** Gradient hội tụ chỉ cần gradient **ước lượng không thiên lệch** (unbiased estimate). Bất kỳ tập con ngẫu nhiên nào của dữ liệu đều cho ước lượng không thiên lệch của gradient toàn bộ.

Thay tổng $\sum_{n=1}^N$ bằng tổng trên **mini-batch** ngẫu nhiên kích thước $M \ll N$.

**Trường hợp cực đoan:** mini-batch = 1 điểm dữ liệu → **SGD thuần túy**.

### Đánh đổi kích thước mini-batch

| Mini-batch lớn | Mini-batch nhỏ |
|---|---|
| Ước lượng gradient chính xác | Ước lượng nhanh, rẻ |
| Hội tụ ổn định hơn | Nhiễu cao → thoát khỏi cực tiểu địa phương |
| Tận dụng tốt vectorization (GPU) | Ít tốn bộ nhớ |
| Mỗi bước tốn kém hơn | Cập nhật thường xuyên hơn |

> **Insight thực tế quan trọng:** Trong ML, mục tiêu là **generalization** (Chương 8), không phải tối thiểu hóa chính xác hàm huấn luyện. Do đó **không cần gradient chính xác** — SGD với mini-batch là đủ và hiệu quả hơn nhiều. Đây là lý do SGD là thuật toán huấn luyện mặc định cho deep learning.

---

## Tổng Kết Mục 7.1

```
Gradient Descent
├── Batch GD: toàn bộ dữ liệu, chính xác nhưng chậm với dữ liệu lớn
├── Mini-batch SGD: tập con ngẫu nhiên, cân bằng tốc độ và độ chính xác
└── SGD: 1 điểm/bước, nhanh nhất, nhiều noise nhất

Cải tiến:
├── Adaptive step-size: tự điều chỉnh γ theo tình hình
└── Momentum: thêm bộ nhớ, giảm zigzag, tăng tốc hội tụ
```

| Vấn đề | Giải pháp |
|---|---|
| Step-size cố định không tối ưu | Adaptive step-size |
| Zigzag trong thung lũng hẹp | Momentum |
| Dữ liệu quá lớn | Mini-batch SGD |
| Kẹt tại cực tiểu địa phương | Mini-batch nhỏ (nhiễu giúp thoát ra) |

# 7.2 Tối Ưu Có Ràng Buộc và Nhân Tử Lagrange

---

## Bài Toán Tối Ưu Có Ràng Buộc

$$\min_\mathbf{x} f(\mathbf{x}) \quad \text{subject to } g_i(\mathbf{x}) \leq 0 \text{ với mọi } i = 1,\ldots,m \tag{7.17}$$

> **Ý tưởng trực quan:** Thay vì tìm cực tiểu tự do, ta tìm cực tiểu **trong vùng cho phép** xác định bởi các ràng buộc $g_i(\mathbf{x}) \leq 0$.

---

## Từ Hàm Chỉ Thị đến Lagrangian

**Cách ngây thơ** — dùng hàm chỉ thị vô hạn để phạt vi phạm ràng buộc:

$$J(\mathbf{x}) = f(\mathbf{x}) + \sum_{i=1}^m \mathbf{1}(g_i(\mathbf{x})), \quad \mathbf{1}(z) = \begin{cases}0 & z \leq 0 \\ \infty & \text{otherwise}\end{cases} \tag{7.18, 7.19}$$

Về lý thuyết đúng nhưng **không tối ưu được** vì hàm không liên tục.

**Giải pháp:** Thay hàm bậc thang vô hạn bằng **hàm tuyến tính** — đây chính là ý tưởng của **nhân tử Lagrange** (Lagrange multiplier).

**Lagrangian:**

$$L(\mathbf{x}, \boldsymbol{\lambda}) = f(\mathbf{x}) + \sum_{i=1}^m \lambda_i g_i(\mathbf{x}) = f(\mathbf{x}) + \boldsymbol{\lambda}^\top \mathbf{g}(\mathbf{x}) \tag{7.20}$$

với $\lambda_i \geq 0$ là các nhân tử Lagrange (dual variables).

> **Tại sao $\lambda_i \geq 0$?** Khi $g_i(\mathbf{x}) > 0$ (vi phạm ràng buộc), hạng tử $\lambda_i g_i(\mathbf{x}) > 0$ tăng giá trị Lagrangian — phạt vi phạm. Nếu $\lambda_i < 0$ thì phạt sẽ thưởng vi phạm, vô nghĩa.

---

## Đối Ngẫu Lagrange (Lagrangian Duality)

### Bài toán gốc (Primal Problem)

$$\min_\mathbf{x} \max_{\boldsymbol{\lambda} \geq 0} L(\mathbf{x}, \boldsymbol{\lambda}) \tag{7.26}$$

**Chứng minh tương đương với (7.17):** Với $\mathbf{x}$ cố định, $\max_{\boldsymbol{\lambda} \geq 0} L(\mathbf{x}, \boldsymbol{\lambda})$:
- Nếu $g_i(\mathbf{x}) > 0$ (vi phạm): đặt $\lambda_i \to \infty$ → $L \to \infty$ = bị loại
- Nếu $g_i(\mathbf{x}) \leq 0$ (thỏa mãn): đặt $\lambda_i = 0$ → $L = f(\mathbf{x})$

Vậy $\max_{\boldsymbol{\lambda} \geq 0} L(\mathbf{x}, \boldsymbol{\lambda}) = J(\mathbf{x})$ — đúng với bài toán gốc.

### Bài toán đối ngẫu (Dual Problem)

$$\max_{\boldsymbol{\lambda} \geq 0} D(\boldsymbol{\lambda}), \quad D(\boldsymbol{\lambda}) = \min_{\mathbf{x} \in \mathbb{R}^d} L(\mathbf{x}, \boldsymbol{\lambda}) \tag{7.22}$$

**Đổi thứ tự min và max** → bài toán **dễ hơn** vì:
- Bài toán trong ($\min_\mathbf{x} L$) là **không có ràng buộc** với $\boldsymbol{\lambda}$ cố định
- $D(\boldsymbol{\lambda})$ là cực tiểu điểm của các hàm affine theo $\boldsymbol{\lambda}$ → **$D$ lõm (concave)** dù $f, g_i$ có thể không lồi
- Bài toán ngoài ($\max_{\boldsymbol{\lambda}} D$) là **tối đa hàm lõm** → luôn giải được hiệu quả

### Đối Ngẫu Yếu (Weak Duality)

Từ bất đẳng thức minimax:

$$\max_{\mathbf{y}} \min_\mathbf{x} \varphi(\mathbf{x},\mathbf{y}) \leq \min_\mathbf{x} \max_\mathbf{y} \varphi(\mathbf{x},\mathbf{y}) \tag{7.23}$$

Áp dụng cho Lagrangian:

$$\underbrace{\max_{\boldsymbol{\lambda} \geq 0} \min_\mathbf{x} L(\mathbf{x},\boldsymbol{\lambda})}_{\text{dual}} \leq \underbrace{\min_\mathbf{x} \max_{\boldsymbol{\lambda} \geq 0} L(\mathbf{x},\boldsymbol{\lambda})}_{\text{primal}} \tag{7.27}$$

> **Ý nghĩa:** Giá trị tối ưu dual **luôn ≤** giá trị tối ưu primal. Dual problem cho **lower bound** của primal. Khi đẳng thức xảy ra ($\text{dual} = \text{primal}$), ta gọi là **strong duality** — cho phép giải primal thông qua dual.

---

## Ràng Buộc Đẳng Thức

Bài toán tổng quát hơn với cả ràng buộc bất đẳng thức và đẳng thức:

$$\min_\mathbf{x} f(\mathbf{x}) \quad \text{s.t.} \quad g_i(\mathbf{x}) \leq 0,\ i=1,\ldots,m \quad \text{và} \quad h_j(\mathbf{x}) = 0,\ j=1,\ldots,n \tag{7.28}$$

**Xử lý ràng buộc đẳng thức:** Thay $h_j(\mathbf{x}) = 0$ bằng hai bất đẳng thức:
$$h_j(\mathbf{x}) \leq 0 \quad \text{và} \quad h_j(\mathbf{x}) \geq 0$$

Kết quả: nhân tử Lagrange của ràng buộc đẳng thức **không bị ràng buộc dấu** (unconstrained), còn nhân tử của bất đẳng thức vẫn $\geq 0$.

**Lagrangian đầy đủ:**

$$L(\mathbf{x}, \boldsymbol{\lambda}, \boldsymbol{\nu}) = f(\mathbf{x}) + \sum_{i=1}^m \lambda_i g_i(\mathbf{x}) + \sum_{j=1}^n \nu_j h_j(\mathbf{x})$$

với $\lambda_i \geq 0$ và $\nu_j \in \mathbb{R}$ tự do.

---

## Cách Giải Bài Toán Lagrange

**Quy trình tổng quát** (khi $f, g_i$ khả vi):

1. Viết Lagrangian $L(\mathbf{x}, \boldsymbol{\lambda})$
2. Lấy đạo hàm theo $\mathbf{x}$, đặt bằng 0: $\frac{\partial L}{\partial \mathbf{x}} = 0$
3. Giải tìm $\mathbf{x}^*$ theo $\boldsymbol{\lambda}$
4. Thay vào $L$ để tính $D(\boldsymbol{\lambda})$
5. Giải bài toán dual: $\max_{\boldsymbol{\lambda} \geq 0} D(\boldsymbol{\lambda})$

Các ví dụ cụ thể sẽ được trình bày trong Mục 7.3.1 và 7.3.2 (khi $f, g_i$ lồi).

---

## Tổng Kết Mục 7.2

```
Bài toán có ràng buộc
        ↓
Thêm nhân tử Lagrange λᵢ ≥ 0
        ↓
Lagrangian L(x, λ) = f(x) + λᵀg(x)
        ↓
Primal: min_x max_{λ≥0} L    (= bài toán gốc)
        ↓  (minimax inequality)
Dual:   max_{λ≥0} min_x L    (dễ giải hơn)
```

| Khái niệm | Ý nghĩa |
|---|---|
| Nhân tử Lagrange $\lambda_i \geq 0$ | "Giá" của ràng buộc $i$ |
| Dual function $D(\boldsymbol{\lambda})$ | Lower bound của primal, luôn lõm |
| Weak duality | Dual $\leq$ Primal |
| Strong duality | Dual $=$ Primal (khi hàm lồi, Mục 7.3) |
| $\lambda_i$ ràng buộc đẳng thức | Không ràng buộc dấu |

# 7.3 Tối Ưu Lồi (Convex Optimization)

---

## Tại Sao Tối Ưu Lồi Quan Trọng?

Với bài toán tối ưu lồi: **mọi cực tiểu địa phương đều là cực tiểu toàn cục** — ta không cần lo về điểm xuất phát hay kẹt tại cực tiểu địa phương. Ngoài ra, **strong duality** đảm bảo nghiệm primal và dual trùng nhau.

---

## Tập Lồi và Hàm Lồi

### Định nghĩa 7.2 — Tập Lồi (Convex Set)

Tập $C$ là **tập lồi** nếu với mọi $\mathbf{x}, \mathbf{y} \in C$ và $\theta \in [0,1]$:

$$\theta\mathbf{x} + (1-\theta)\mathbf{y} \in C \tag{7.29}$$

> **Trực quan:** Đoạn thẳng nối bất kỳ hai điểm nào trong tập đều nằm trong tập. Hình tròn là tập lồi; hình trăng lưỡi liềm không phải.

### Định nghĩa 7.3 — Hàm Lồi (Convex Function)

Hàm $f: \mathbb{R}^D \to \mathbb{R}$ là **hàm lồi** nếu với mọi $\mathbf{x}, \mathbf{y}$ trong miền xác định và $\theta \in [0,1]$:

$$f(\theta\mathbf{x} + (1-\theta)\mathbf{y}) \leq \theta f(\mathbf{x}) + (1-\theta)f(\mathbf{y}) \tag{7.30}$$

> **Trực quan:** Đoạn thẳng nối bất kỳ hai điểm trên đồ thị $f$ luôn nằm **phía trên** đồ thị — hàm có dạng "bát lõm lên". Hàm lõm (concave) là phủ định của hàm lồi.

**Epigraph:** Tập hợp các điểm "phía trên" đồ thị hàm lồi — $\{(x, t): f(x) \leq t\}$ — là một tập lồi.

### Hai Cách Kiểm Tra Lồi Tiện Lợi

**Với hàm khả vi** — điều kiện gradient bậc nhất:

$$f(\mathbf{y}) \geq f(\mathbf{x}) + \nabla_\mathbf{x}f(\mathbf{x})^\top(\mathbf{y} - \mathbf{x}) \quad \forall \mathbf{x}, \mathbf{y} \tag{7.31}$$

Tiếp tuyến tại bất kỳ điểm nào đều nằm **phía dưới** đồ thị hàm lồi.

**Với hàm khả vi hai lần** — điều kiện Hessian bậc hai:

$$f \text{ lồi} \iff \nabla^2_\mathbf{x}f(\mathbf{x}) \succeq 0 \quad \text{(nửa xác định dương)}$$

### Ví dụ 7.3 — Entropy âm là hàm lồi

$f(x) = x\log_2 x$ với $x > 0$.

**Kiểm tra theo định nghĩa** tại $x=2$, $y=4$, $\theta=0.5$:
- VT: $f(0.5 \cdot 2 + 0.5 \cdot 4) = f(3) = 3\log_2 3 \approx 4.75$
- VP: $0.5f(2) + 0.5f(4) = 0.5(2) + 0.5(8) = 5$
- $4.75 \leq 5$ ✓

**Kiểm tra theo gradient** (7.31) tại $x=2$, $y=4$:

$$\nabla_x(x\log_2 x) = \log_2 x + \frac{1}{\ln 2} \tag{7.32}$$

- VT: $f(4) = 8$
- VP: $f(2) + \nabla f(2) \cdot (4-2) = 2 + \left(1 + \frac{1}{\ln 2}\right) \cdot 2 \approx 6.9$
- $8 \geq 6.9$ ✓

### Tính Chất Bảo Toàn Lồi (Ví dụ 7.4)

**Tổ hợp tuyến tính không âm của hàm lồi vẫn lồi:** Nếu $f_1, f_2$ lồi và $\alpha, \beta \geq 0$:

$$\alpha f_1(\mathbf{x}) + \beta f_2(\mathbf{x}) \text{ lồi} \tag{7.34-7.37}$$

**Bất đẳng thức Jensen:** Dạng tổng quát của định nghĩa lồi — tổ hợp lồi không âm của hàm lồi:

$$f\!\left(\sum_i \theta_i \mathbf{x}_i\right) \leq \sum_i \theta_i f(\mathbf{x}_i), \quad \theta_i \geq 0,\ \sum_i\theta_i = 1 \tag{7.30}$$

---

## Bài Toán Tối Ưu Lồi

$$\min_\mathbf{x} f(\mathbf{x}) \quad \text{s.t.} \quad g_i(\mathbf{x}) \leq 0,\ i=1,\ldots,m \quad \text{và} \quad h_j(\mathbf{x}) = 0,\ j=1,\ldots,n \tag{7.38}$$

là **bài toán tối ưu lồi** khi $f, g_i$ là hàm lồi và $h_j = 0$ là tập lồi.

---

## 7.3.1 Quy Hoạch Tuyến Tính (Linear Programming)

Trường hợp đặc biệt khi **tất cả hàm đều tuyến tính**:

$$\min_{\mathbf{x} \in \mathbb{R}^d} \mathbf{c}^\top\mathbf{x} \quad \text{s.t.} \quad A\mathbf{x} \leq \mathbf{b} \tag{7.39}$$

**Lagrangian:**

$$L(\mathbf{x}, \boldsymbol{\lambda}) = \mathbf{c}^\top\mathbf{x} + \boldsymbol{\lambda}^\top(A\mathbf{x} - \mathbf{b}) = (\mathbf{c} + A^\top\boldsymbol{\lambda})^\top\mathbf{x} - \boldsymbol{\lambda}^\top\mathbf{b} \tag{7.40, 7.41}$$

Đặt $\frac{\partial L}{\partial \mathbf{x}} = 0$: $\mathbf{c} + A^\top\boldsymbol{\lambda} = 0$ → dual function $D(\boldsymbol{\lambda}) = -\boldsymbol{\lambda}^\top\mathbf{b}$.

**Bài toán dual:**

$$\max_{\boldsymbol{\lambda} \in \mathbb{R}^m} -\mathbf{b}^\top\boldsymbol{\lambda} \quad \text{s.t.} \quad \mathbf{c} + A^\top\boldsymbol{\lambda} = 0,\ \boldsymbol{\lambda} \geq 0 \tag{7.43}$$

Dual cũng là linear program nhưng với $m$ biến thay vì $d$ biến → **chọn giải primal hay dual tùy cái nào nhỏ hơn**.

---

## 7.3.2 Quy Hoạch Toàn Phương (Quadratic Programming)

Hàm mục tiêu bậc hai lồi, ràng buộc affine:

$$\min_{\mathbf{x} \in \mathbb{R}^d} \frac{1}{2}\mathbf{x}^\top Q\mathbf{x} + \mathbf{c}^\top\mathbf{x} \quad \text{s.t.} \quad A\mathbf{x} \leq \mathbf{b} \tag{7.45}$$

với $Q \in \mathbb{R}^{d \times d}$ xác định dương.

**Lagrangian:**

$$L(\mathbf{x}, \boldsymbol{\lambda}) = \frac{1}{2}\mathbf{x}^\top Q\mathbf{x} + (\mathbf{c} + A^\top\boldsymbol{\lambda})^\top\mathbf{x} - \boldsymbol{\lambda}^\top\mathbf{b} \tag{7.48}$$

Đặt $\frac{\partial L}{\partial \mathbf{x}} = 0$: $Q\mathbf{x} + (\mathbf{c} + A^\top\boldsymbol{\lambda}) = 0$ → $\mathbf{x}^* = -Q^{-1}(\mathbf{c} + A^\top\boldsymbol{\lambda})$ (vì $Q$ khả nghịch).

**Dual function** (thay $\mathbf{x}^*$ vào $L$):

$$D(\boldsymbol{\lambda}) = -\frac{1}{2}(\mathbf{c} + A^\top\boldsymbol{\lambda})^\top Q^{-1}(\mathbf{c} + A^\top\boldsymbol{\lambda}) - \boldsymbol{\lambda}^\top\mathbf{b} \tag{7.51}$$

**Bài toán dual:**

$$\max_{\boldsymbol{\lambda} \geq 0} -\frac{1}{2}(\mathbf{c} + A^\top\boldsymbol{\lambda})^\top Q^{-1}(\mathbf{c} + A^\top\boldsymbol{\lambda}) - \boldsymbol{\lambda}^\top\mathbf{b} \tag{7.52}$$

> **Ứng dụng:** SVM (Chương 12) là bài toán quy hoạch toàn phương — đây là lý do SVM giải được hiệu quả và có đảm bảo toàn cục.

---

## 7.3.3 Biến Đổi Legendre-Fenchel và Liên Hợp Lồi

### Ý Tưởng Hình Học

Tập lồi có thể mô tả tương đương bằng **các siêu phẳng hỗ trợ** (supporting hyperplanes). Hàm lồi = epigraph = tập lồi → hàm lồi cũng mô tả được bằng tiếp tuyến tại mỗi điểm → mô tả được qua **gradient**.

**Biến đổi Legendre-Fenchel** (hay **convex conjugate**) hình thức hóa ý tưởng này: chuyển hàm $f(x)$ thành hàm của gradient $s = \nabla f(x)$.

### Định Nghĩa 7.4 — Liên Hợp Lồi (Convex Conjugate)

$$f^*(s) = \sup_{\mathbf{x} \in \mathbb{R}^D} \left(\langle s, \mathbf{x}\rangle - f(\mathbf{x})\right) \tag{7.53}$$

**Hình học:** Với gradient $s$ cố định, tìm đường thẳng $y = sx + c$ tiếp xúc với $f$ từ phía dưới. Tung độ gốc $c$ nhỏ nhất của đường thẳng như vậy là $-f^*(s)$.

**Trường hợp lồi khả vi** — không cần $\sup$, tương ứng một-một:

$$f^*(s) = s x_0 - f(x_0) \quad \text{tại điểm } s = \nabla f(x_0) \tag{7.58}$$

> **Tính chất quan trọng:** Áp dụng Legendre transform hai lần cho hàm lồi → trở về hàm gốc. "Độ dốc của $f^*$ là $x$" (đảo chiều của "$f$ có độ dốc $s$").

### Ví dụ 7.7 — Liên Hợp của Hàm Bậc Hai

Cho $f(\mathbf{y}) = \frac{\lambda}{2}\mathbf{y}^\top K^{-1}\mathbf{y}$ với $K$ xác định dương:

$$f^*(\boldsymbol{\alpha}) = \sup_\mathbf{y} \left[\langle\mathbf{y}, \boldsymbol{\alpha}\rangle - \frac{\lambda}{2}\mathbf{y}^\top K^{-1}\mathbf{y}\right] \tag{7.60}$$

Đặt gradient bằng 0: $\boldsymbol{\alpha} - \lambda K^{-1}\mathbf{y} = 0$ → $\mathbf{y}^* = \frac{1}{\lambda}K\boldsymbol{\alpha}$.

Thay vào:

$$f^*(\boldsymbol{\alpha}) = \frac{1}{2\lambda}\boldsymbol{\alpha}^\top K\boldsymbol{\alpha} \tag{7.62}$$

### Ví dụ 7.8 — Liên Hợp của Tổng Hàm

Với $L(\mathbf{t}) = \sum_{i=1}^n \ell_i(t_i)$:

$$L^*(\mathbf{z}) = \sup_\mathbf{t}\left[\langle\mathbf{z},\mathbf{t}\rangle - \sum_i \ell_i(t_i)\right] = \sum_i \sup_{t_i}[z_i t_i - \ell_i(t_i)] = \sum_i \ell_i^*(z_i) \tag{7.63}$$

**Liên hợp của tổng = tổng của các liên hợp** — tính chất cực kỳ tiện lợi khi hàm mất mát tách được theo từng điểm dữ liệu.

### Ví dụ 7.9 — Đối Ngẫu qua Legendre-Fenchel

Cho $f(\mathbf{y})$ và $g(\mathbf{x})$ lồi, ràng buộc $A\mathbf{x} = \mathbf{y}$:

$$\min_\mathbf{x} f(A\mathbf{x}) + g(\mathbf{x}) = \max_\mathbf{u} -f^*(\mathbf{u}) - g^*(-A^\top\mathbf{u}) \tag{7.68}$$

Đây là cách **tự động tìm bài toán dual** từ primal thông qua convex conjugate — không cần "đoán" dạng dual.

---

## Tổng Kết Mục 7.3

| Khái niệm | Định nghĩa | Kiểm tra |
|---|---|---|
| Tập lồi | Đoạn thẳng giữa 2 điểm nằm trong tập | Định nghĩa trực tiếp |
| Hàm lồi | Đồ thị nằm dưới đoạn thẳng nối 2 điểm | Gradient (7.31) hoặc Hessian $\succeq 0$ |
| LP | $\mathbf{c}^\top\mathbf{x}$, ràng buộc tuyến tính | Dual cũng là LP |
| QP | $\frac{1}{2}\mathbf{x}^\top Q\mathbf{x}$, $Q \succ 0$ | Dual là QP với biến $\boldsymbol{\lambda}$ |
| Convex conjugate | $f^*(s) = \sup_x[sx - f(x)]$ | Công cụ tự động tìm dual |

**Tại sao tối ưu lồi đặc biệt:**
- Cực tiểu địa phương = cực tiểu toàn cục
- Strong duality: primal = dual
- Nhiều hàm mục tiêu ML được thiết kế để lồi (SVM, logistic regression,...)
- Convex conjugate cung cấp framework thống nhất để tìm dual problems